# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 4096
USE_SFT_ADAPTER = False
SFT_ADAPTER_DIR = "checkpoints/qwen3_4b_sft_lora_800step"
RUN_LIMIT   = 10     # Use a small integer for quick experiments, or None for all questions
NUM_SELF_CONSISTENCY = 1
USE_REFLECTION = False  # The first subset run showed reflection often damaged good consensus answers
EXPERIMENT_VERSION = "v4_baseline_prompt_robust_extract"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import re
import sys
from collections import Counter
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use problem-type-specific prompts and enforce a clean final-answer contract:

- **MCQ** — eliminate wrong choices, then output `Final Answer: A`
- **Free-form** — solve symbolically/numerically, then output `Final Answer: ...`
- **Multi-answer** — every `[ANS]` placeholder gets one ordered value: `Final Answer: [answer1, answer2, ...]`
- **Reflection** — a second pass verifies arithmetic, algebra, interpretation, and formatting

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_FREE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

SYSTEM_PROMPT_REFLECT = (
    "You are verifying a proposed math answer. Check arithmetic, algebra, interpretation of the question, "
    "and final-answer formatting. Then output only the corrected final answer using the required format."
)

# Original starter baseline prompts, kept for controlled subset comparisons.
BASELINE_SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

BASELINE_SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_FREE, question


def build_baseline_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Original starter prompt for baseline comparisons."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return BASELINE_SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return BASELINE_SYSTEM_PROMPT_MATH, question


def build_reflection_prompt(item: dict, proposed_answer: str) -> tuple[str, str]:
    """Return a verification prompt for a proposed final answer."""
    _, user_prompt = build_prompt(item["question"], item.get("options"))
    if item.get("options"):
        required = "Final Answer: <one option letter>"
    else:
        required = "Final Answer: \\boxed{...} or Final Answer: \\boxed{answer1, answer2, ...}"
    user = (
        f"Problem:\n{user_prompt}\n\n"
        f"Proposed answer:\n{proposed_answer}\n\n"
        f"Verify it carefully. Output only the corrected answer in this format: {required}"
    )
    return SYSTEM_PROMPT_REFLECT, user


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR if USE_SFT_ADAPTER else MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

if USE_SFT_ADAPTER:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    llm = PeftModel.from_pretrained(base_model, SFT_ADAPTER_DIR)
    llm.eval()
    INFERENCE_BACKEND = "transformers_sft_lora"
else:
    llm = LLM(
        model=MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        enable_prefix_caching=False,
        gpu_memory_utilization=0.50,
        max_model_len=16384,
        trust_remote_code=True,
        max_num_seqs=256,
        max_num_batched_tokens=32768,
    )
    INFERENCE_BACKEND = "vllm_base"

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

reflection_sampling_params = SamplingParams(
    max_tokens=1024,
    temperature=0.0,
    top_p=1.0,
    repetition_penalty=1.0,
)

print(f"Model loaded with backend: {INFERENCE_BACKEND}")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
data_to_run = data if RUN_LIMIT is None else data[:RUN_LIMIT]


def make_chat_prompt(system: str, user: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )


def build_generation_prompts(items, prompt_builder=build_prompt, num_samples=NUM_SELF_CONSISTENCY):
    prompts = []
    prompt_items = []
    for item in items:
        system, user = prompt_builder(item["question"], item.get("options"))
        for sample_idx in range(num_samples):
            prompts.append(make_chat_prompt(system, user))
            prompt_items.append((item, sample_idx))
    return prompts, prompt_items


def generate_texts(prompts, params=None):
    if INFERENCE_BACKEND == "vllm_base":
        outputs = llm.generate(prompts, sampling_params=params)
        return [out.outputs[0].text.strip() for out in outputs]

    texts = []
    for prompt in tqdm(prompts, desc="Generating"):
        inputs = tokenizer(
            [prompt],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=16384,
        ).to(llm.device)
        gen_kwargs = {
            "max_new_tokens": MAX_TOKENS,
            "do_sample": NUM_SELF_CONSISTENCY > 1,
            "pad_token_id": tokenizer.eos_token_id,
        }
        if NUM_SELF_CONSISTENCY > 1:
            gen_kwargs.update({"temperature": 0.6, "top_p": 0.95, "top_k": 20})
        with torch.no_grad():
            output_ids = llm.generate(**inputs, **gen_kwargs)
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        texts.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return texts


print(f"Running original baseline on {len(data_to_run)} questions...")
baseline_prompts, baseline_prompt_items = build_generation_prompts(
    data_to_run,
    prompt_builder=build_baseline_prompt,
    num_samples=1,
)
baseline_responses = generate_texts(baseline_prompts, sampling_params)

print(f"Generating improved {NUM_SELF_CONSISTENCY} sample(s) for {len(data_to_run)} questions with {INFERENCE_BACKEND}...")
if NUM_SELF_CONSISTENCY == 1:
    # Improved pipeline currently keeps the measured-best baseline prompt and improves extraction/scoring.
    # Reuse the baseline generations so the comparison isolates pipeline changes instead of sampling noise.
    prompts, prompt_items = baseline_prompts, baseline_prompt_items
    candidate_texts = baseline_responses
else:
    prompts, prompt_items = build_generation_prompts(data_to_run)
    candidate_texts = generate_texts(prompts, sampling_params)

candidate_responses = {item.get("id"): [] for item in data_to_run}
for (item, sample_idx), text in zip(prompt_items, candidate_texts):
    candidate_responses[item.get("id")].append(text)

print(f"Generated {len(candidate_texts)} candidate responses.")

### Generate with Transformers (for Datahub)

In [ ]:
# Optional Transformers path. Run this cell instead of the vLLM generation cell above
# after loading a Transformers model named `llm` and importing torch.
#
# data_to_run = data if RUN_LIMIT is None else data[:RUN_LIMIT]
# prompts, prompt_items = build_generation_prompts(data_to_run)
# candidate_texts = []
#
# print(f"Generating {len(prompts)} candidate responses with Transformers...")
# for prompt in tqdm(prompts, desc="Generating"):
#     inputs = tokenizer(
#         [prompt],
#         return_tensors="pt",
#         padding=True,
#         truncation=True,
#         max_length=16384,
#     ).to(llm.device)
#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             repetition_penalty=1.0,
#             do_sample=True,
#         )
#     new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
#     candidate_texts.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
#
# candidate_responses = {item.get("id"): [] for item in data_to_run}
# for (item, sample_idx), text in zip(prompt_items, candidate_texts):
#     candidate_responses[item.get("id")].append(text)

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `Final Answer: ...` and compare to the gold letter.
- **Free-form**: extract `Final Answer: ...`, then use `Judger.auto_judge()` for symbolic and numeric equivalence.
- **Self-consistency**: vote over extracted final answers before scoring.

Each public-set result record contains `{id, is_mcq, gold, response, candidates, vote_counts, correct}`.

In [ ]:
def extract_final_answer(text: str) -> str:
    """Extract the final answer after our prompt contract, with baseline fallbacks."""
    if not text:
        return ""

    text = text.replace("</think>", "").strip()
    boxed = re.findall(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", text)
    if boxed:
        return boxed[-1].strip()

    matches = re.findall(r"(?im)^\s*(?:\*\*)?\s*final\s*answer\s*(?:\*\*)?\s*:\s*(.+?)\s*$", text)
    if matches:
        ans = matches[-1].strip()
        ans = re.split(r"\n\s*\n", ans, maxsplit=1)[0].strip()
        return ans.strip().strip("$ ")

    nonempty = [line.strip() for line in text.splitlines() if line.strip()]
    return nonempty[-1] if nonempty else text.strip()


def extract_letter_strict(text: str) -> str:
    final = extract_final_answer(text).strip().strip("$ ")
    m = re.fullmatch(r"(?:option\s*)?([A-Ja-j])", final)
    if m:
        return m.group(1).upper()
    m = re.fullmatch(r"\\boxed\{\s*([A-Ja-j])\s*\}", final)
    if m:
        return m.group(1).upper()
    return ""


def normalize_free_answer(answer: str, item: dict) -> str:
    answer = answer.strip().strip("$ ")
    m = re.fullmatch(r"\[\s*(.+?)\s*\]", answer)
    if m:
        return m.group(1).strip()
    m = re.fullmatch(r"\\boxed\{\s*(.+?)\s*\}", answer)
    if m:
        return m.group(1).strip()
    return answer


def normalize_vote_key(answer: str, is_mcq: bool) -> str:
    if is_mcq:
        return extract_letter_strict(answer)
    answer = answer.strip().strip("$ ")
    answer = re.sub(r"^\\boxed\{(.+)\}$", r"\1", answer)
    answer = re.sub(r"\s+", "", answer)
    return answer.lower()


def extract_letter(text: str) -> str:
    """Extract MCQ letters without treating arbitrary capital letters as answers."""
    text = text or ""
    final = extract_final_answer(text).strip().strip("$ ")
    m = re.fullmatch(r"(?:option\s*)?([A-Ja-j])", final)
    if m:
        return m.group(1).upper()
    patterns = [
        r"\\boxed\{\s*([A-Ja-j])\s*\}",
        r"(?:final\s*)?answer\s*(?:for\s*(?:question|part)\s*\d+)?\s*(?:is|:|=)\s*(?:option\s*)?([A-Ja-j])\b",
        r"(?:choose|select|option)\s+([A-Ja-j])\b",
        r"\b([A-Ja-j])\s*(?:is\s+)?(?:the\s+)?(?:correct|best)\s+(?:answer|option)\b",
    ]
    matches = []
    for pattern in patterns:
        matches.extend(re.findall(pattern, text, flags=re.IGNORECASE))
    return matches[-1].upper() if matches else ""


def choose_consensus(candidates: list[str], is_mcq: bool) -> tuple[str, dict]:
    finals = [extract_final_answer(c) for c in candidates]
    if is_mcq:
        letters = [extract_letter(c) for c in candidates]
        votes = [letter for letter in letters if letter]
        if votes:
            winner_key, _ = Counter(votes).most_common(1)[0]
            return f"\\boxed{{{winner_key}}}", {"finals": finals, "vote_counts": dict(Counter(votes))}
        return candidates[0] if candidates else "", {"finals": finals, "vote_counts": {}}
    keys = [normalize_vote_key(ans, is_mcq) for ans in finals]
    nonempty_keys = [k for k in keys if k]
    if not nonempty_keys:
        return finals[0] if finals else "", {"finals": finals, "vote_counts": {}}
    winner_key, _ = Counter(nonempty_keys).most_common(1)[0]
    winner_idx = next(i for i, key in enumerate(keys) if key == winner_key)
    return finals[winner_idx], {"finals": finals, "vote_counts": dict(Counter(nonempty_keys))}


def final_answer_line(answer: str, is_mcq: bool, item: Optional[dict] = None) -> str:
    if is_mcq:
        letter = extract_letter_strict(answer) or extract_letter(answer)
        return f"Final Answer: {letter}" if letter else f"Final Answer: {answer.strip()}"
    if item is not None:
        answer = normalize_free_answer(answer, item)
    return f"Final Answer: \\boxed{{{answer.strip()}}}"


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


def extract_letter_legacy(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq_legacy(response: str, gold_letter: str) -> bool:
    return extract_letter_legacy(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

def score_items(items, scored_responses, metadata_list=None, label="run", legacy_mcq=False):
    if metadata_list is None:
        metadata_list = [{} for _ in scored_responses]
    scored = []
    for item, response, metadata in tqdm(
        zip(items, scored_responses, metadata_list),
        total=len(items),
        desc=f"Scoring {label}",
    ):
        is_mcq = bool(item.get("options"))
        gold = item["answer"]

        if is_mcq:
            correct = score_mcq_legacy(response, str(gold)) if legacy_mcq else score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        scored.append({
            "id": item.get("id"),
            "is_mcq": is_mcq,
            "gold": gold,
            "response": response,
            "candidates": metadata.get("finals", []),
            "vote_counts": metadata.get("vote_counts", {}),
            "correct": correct,
        })
    return scored


baseline_results = score_items(data_to_run, baseline_responses, label="baseline", legacy_mcq=True)

responses = []
result_metadata = []
for item in data_to_run:
    is_mcq = bool(item.get("options"))
    consensus_answer, metadata = choose_consensus(candidate_responses[item.get("id")], is_mcq)
    responses.append(final_answer_line(consensus_answer, is_mcq, item))
    result_metadata.append(metadata)

if USE_REFLECTION:
    reflection_prompts = []
    for item, answer in zip(data_to_run, responses):
        system, user = build_reflection_prompt(item, answer)
        reflection_prompts.append(make_chat_prompt(system, user))
    print(f"Running reflection pass for {len(reflection_prompts)} questions...")
    reflected_texts = generate_texts(reflection_prompts, reflection_sampling_params)
    responses = [
        final_answer_line(extract_final_answer(text), bool(item.get("options")), item)
        for item, text in zip(data_to_run, reflected_texts)
    ]

results = score_items(data_to_run, responses, result_metadata, label="improved")

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]
baseline_mcq_res  = [r for r in baseline_results if r["is_mcq"]]
baseline_free_res = [r for r in baseline_results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

experiment_row = {
    "version": EXPERIMENT_VERSION,
    "change": f"type prompts + final-answer extraction + self-consistency={NUM_SELF_CONSISTENCY} + reflection={USE_REFLECTION}",
    "num_questions": len(results),
    "mcq_accuracy": acc(mcq_res),
    "free_form_accuracy": acc(free_res),
    "overall_accuracy": acc(results),
    "baseline_overall_accuracy": acc(baseline_results),
}

comparison_rows = [
    ("baseline", baseline_results, baseline_mcq_res, baseline_free_res),
    ("improved", results, mcq_res, free_res),
]

print("=" * 50)
print(f"SUBSET EVALUATION RESULTS ({len(results)} questions)")
print("=" * 50)
for name, all_res, mcq_part, free_part in comparison_rows:
    print(f"{name.upper():9s} | MCQ {sum(r['correct'] for r in mcq_part):2d}/{len(mcq_part):2d} ({acc(mcq_part):6.2f}%) | "
          f"Free {sum(r['correct'] for r in free_part):2d}/{len(free_part):2d} ({acc(free_part):6.2f}%) | "
          f"Overall {sum(r['correct'] for r in all_res):2d}/{len(all_res):2d} ({acc(all_res):6.2f}%)")
print("-" * 50)
print("Changed outcomes:")
for base, new in zip(baseline_results, results):
    if base["correct"] != new["correct"]:
        direction = "fixed" if new["correct"] else "regressed"
        print(f"  id={new['id']}: {direction} | baseline={extract_final_answer(base['response'])!r} | improved={extract_final_answer(new['response'])!r} | gold={new['gold']!r}")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "candidates": r.get("candidates", []),
                      "vote_counts": r.get("vote_counts", {}), "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} improved records to {out_path}")

baseline_out_path = out_path.with_name(out_path.stem + "_baseline_subset.jsonl")
with open(baseline_out_path, "w") as f:
    for r in baseline_results:
        record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                  "response": r["response"], "correct": r["correct"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(baseline_results)} baseline records to {baseline_out_path}")

experiment_path = out_path.parent / "experiment_log.jsonl"
with open(experiment_path, "a") as f:
    f.write(json.dumps(experiment_row) + "\n")

print(f"Appended experiment summary to {experiment_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!